In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import mode
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
import ta.volume
import ta.momentum
import ta.volatility
import ta.trend
from advanced_ta import LorentzianClassification
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report
from datetime import timedelta,datetime
from decimal import Decimal

In [ ]:
class BinaryArrowSignalPredictor:
    def __init__(self, df: pd.DataFrame) -> None:
        self.df = df.copy()
        # Define parameters
        self.SignalGap = 20
        self.BarsToCount = len(df)
        self.dist = 24
        decimal_places = len(str(self.df['close'].iloc[0]).split('.')[-1])
        self.Point = float(Decimal(f"0.{'0' * (decimal_places - 1)}1"))  # Assuming Point is 0.00001 for Forex pairs, adjust as needed

        # Initialize columns for signals
        self.df.loc[:,'BinaryArrow'] = 0

    def ihighest(self, highs: pd.Series, period: int, shift: int) -> int:
        """Find the index of the highest high in a given range."""
        if shift < 0 or shift + period > len(highs):
            return np.nan
        return highs.iloc[shift:shift + period].idxmax()

    def ilowest(self, lows: pd.Series, period: int, shift: int) -> int:
        """Find the index of the lowest low in a given range."""
        if shift < 0 or shift + period > len(lows):
            return np.nan
        return lows.iloc[shift:shift + period].idxmin()

    def run(self) -> pd.DataFrame:
        """Calculate buy and sell signals and update DataFrame."""
        high_series = self.df['high']
        low_series = self.df['low']
        
        # Calculate highest high and lowest low
        for i in range(self.BarsToCount,1, -1):
            hhb = self.ihighest(high_series, self.dist, i - self.dist // 2)
            llb = self.ilowest(low_series, self.dist, i - self.dist // 2)
            
            if pd.notna(hhb) and i == hhb:
                self.df.at[self.df.index[i], 'BinaryArrow'] = 2
            if pd.notna(llb) and i == llb:
                self.df.at[self.df.index[i], 'BinaryArrow'] = 1

        return self.df


In [ ]:
# class ExtremeBinarySignalPredictor:
#     def __init__(self,df: pd.DataFrame,df1min: pd.DataFrame, file_path=None, Gd_228=25.0, Gi_220=17, EnableAlert=True, SoundFilename="alert.wav"):
#         self.file_path = file_path
#         self.Gd_228 = Gd_228
#         self.Gi_220 = Gi_220
#         self.EnableAlert = EnableAlert
#         self.SoundFilename = SoundFilename
#         self.data = df
#         self.data_1_min = df1min
#         self.Gd_248 = self.determine_Gd_248()

#     def load_data(self):
#         data = pd.read_csv(self.file_path)
#         data['datetime'] = pd.to_datetime(data['datetime'])
#         return data

#     def determine_Gd_248(self):
#         return 0.0001 if self.data['close'].apply(lambda x: len(str(x).split('.')[1]) < 4).all() else 0.00001

#     def calculate_indicators(self):
#         data = self.data
#         data_1_min = self.data_1_min
#         data['ATR'] = ta.volatility.AverageTrueRange(data['high'], data['low'], data['close'], window=5).average_true_range()
#         data['Stochastic'] = ta.momentum.StochasticOscillator(data['high'], data['low'], data['close'], window=20, smooth_window=12).stoch()
#         data['CCI'] = ta.trend.CCIIndicator(data_1_min['high'], data_1_min['low'], data_1_min['close'], window=80).cci()
#         data['Momentum_60'] = ta.momentum.ROCIndicator(data_1_min['close'], window=60).roc()
#         data['Momentum_4'] = ta.momentum.ROCIndicator(data['close'], window=4).roc()
#         data['WPR'] = ta.momentum.WilliamsRIndicator(data_1_min['high'], data_1_min['low'], data_1_min['close'], lbp=14).williams_r()
#         data['Force'] = ta.volume.ForceIndexIndicator(data['close'], data['volume'], window=13).force_index()
#         bb = ta.volatility.BollingerBands(data['close'], window=20, window_dev=2)
#         data['Bollinger_Upper'] = bb.bollinger_hband()
#         data['Bollinger_Lower'] = bb.bollinger_lband()
#         data['MA_High'] = ta.trend.EMAIndicator(data['high'], window=1).ema_indicator()
#         data['MA_Median'] = ta.trend.EMAIndicator((data_1_min['high'] + data_1_min['low']) / 2, window=1).ema_indicator()
#         data['MA_Low'] = ta.trend.EMAIndicator(data_1_min['low'], window=1).ema_indicator()

#     def determine_market_condition(self, stochastic_value):
#         if 75.0 >= stochastic_value >= 25.0:
#             return "SAFE TRADE"
#         elif 88.0 >= stochastic_value > 75.0 or 25.0 > stochastic_value >= 12.0:
#             return "S/R AREA"
#         elif stochastic_value > 88.0 or stochastic_value < 12.0:
#             return "HIGH RISK!"
#         else:
#             return ""

#     def apply_market_conditions(self):
#         self.data['Market_Condition'] = self.data['Stochastic'].apply(self.determine_market_condition)
    
#     def calculate_extreme_buysell(self,row):
#         i = row.name
#         if i == 0:  # Skip the first row to avoid out-of-bounds error
#             return 0
#         if (row['ExtremeBinarySignal'] < row['open'] and 
#             self.data.at[i-1, 'Market_Condition'] in ('SAFE TRADE', "S/R AREA") and 
#             row['Market_Condition'] != 'HIGH RISK!'):
#             return 1
#         elif (row['ExtremeBinarySignal'] > row['open'] and 
#             self.data.at[i-1, 'Market_Condition'] in ('SAFE TRADE', "S/R AREA") and 
#             row['Market_Condition'] != 'HIGH RISK!'):
#             return 2
#         else:
#             return 0

#     def generate_signals(self):
#         data = self.data
#         data['ExtremeBUYSELL'] = 0
#         data['ExtremeBinarySignal'] = np.nan

#         for i in range(10, len(data)):
#             G_high_324 = data['high'][i-10:i].max()
#             G_low_332 = data['low'][i-10:i].min()
#             Gd_316 = sum((10 - j) * (data['high'][i-j] - data['low'][i-j]) for j in range(10)) / 55.0

#             if data['close'][i] > G_high_324 - (G_high_324 - G_low_332) * self.Gi_220 / 100.0:
#                 data.at[i, 'ExtremeBinarySignal'] = data['high'][i] + Gd_316 / 2.0
#             elif data['close'][i] < G_low_332 + (G_high_324 - G_low_332) * self.Gi_220 / 100.0:
#                 data.at[i, 'ExtremeBinarySignal'] = data['low'][i] - Gd_316 / 2.0
#         data['ExtremeBUYSELL'] = data.apply(self.calculate_extreme_buysell, axis=1)
#         return data

#     def adjust_timezones(self):
#         self.data['UTC'] = pd.to_datetime(self.data['datetime']) + timedelta(hours=5)
#         self.data['GMT'] = self.data['UTC'] + timedelta(hours=2)

#     def save_output(self, output_file):
#         self.data.to_csv(output_file, index=False)

#     def run(self, output_file=None) -> pd.DataFrame:
#         self.calculate_indicators()
#         self.apply_market_conditions()
#         return self.generate_signals()
#         # self.adjust_timezones()
#         # self.save_output(output_file)



In [ ]:
# class SuperArrowSignalGenerator:
#     def __init__(self, df: pd.DataFrame,file_path=None):
#         self.df = df.reset_index(drop=True)
#         self.parameters = {
#             'FasterMovingAverage': 5,
#             'SlowerMovingAverage': 12,
#             'RSIPeriod': 12,
#             'MagicFilterPeriod': 1,
#             'BollingerbandsPeriod': 10,
#             'BollingerbandsShift': 0,
#             'BollingerbandsDeviation': 0.5,
#             'BullsPowerPeriod': 50,
#             'BearsPowerPeriod': 50,
#             'Utstup': 10,
#             'Alerts': True
#         }
#         self.initialize_conditions()

#     def initialize_conditions(self):
#         self.Gi_132 = False
#         self.Gi_136 = False
#         self.Gi_140 = False
#         self.Gi_144 = False
#         self.Gi_148 = False
#         self.Gi_152 = False
#         self.Gi_156 = False
#         self.Gi_160 = False
#         self.Gi_164 = False
#         self.Gi_168 = False
#         self.Gi_172 = 0
#         self.Gi_176 = False
#         self.Gi_180 = False

#     def calculate_indicators(self):
#         params = self.parameters
#         self.df['ema_fast'] = ta.trend.ema_indicator(self.df['close'], window=params['FasterMovingAverage'])
#         self.df['ema_slow'] = ta.trend.ema_indicator(self.df['close'], window=params['SlowerMovingAverage'])
#         self.df['rsi'] = ta.momentum.rsi(self.df['close'], window=params['RSIPeriod'])
#         self.df['bulls_power'] = self.df['high'] - ta.trend.ema_indicator(self.df['close'], window=params['BullsPowerPeriod'])
#         self.df['bears_power'] = self.df['low'] - ta.trend.ema_indicator(self.df['close'], window=params['BearsPowerPeriod'])
#         bb = ta.volatility.BollingerBands(self.df['close'], window=params['BollingerbandsPeriod'], window_dev=params['BollingerbandsDeviation'])
#         self.df['bb_upper'] = bb.bollinger_hband()
#         self.df['bb_lower'] = bb.bollinger_lband()

#     def apply_strategy(self):
#         self.df['SuperArrowSignal'] = 0
#         for i in range(len(self.df)-1, 1, -1):
#             if i < 10:
#                 continue  # Skip initial periods where sufficient data isn't available

#             Ld_140 = np.sum(np.abs(self.df['high'][i:i + 10] - self.df['low'][i:i + 10]))
#             Ld_132 = Ld_140 / 10.0
#             Ld_124 = 100 - 100.0 * ((Ld_132 - 0.0) / 10.0)

#             if Ld_124 >= 0.0:
#                 self.Gi_148 = True
#                 self.Gi_168 = False
#             else:
#                 self.Gi_148 = False
#                 self.Gi_168 = True

#             if self.df['close'][i] > self.df['bb_upper'][i] and self.df['close'][i - 1] >= self.df['bb_upper'][i - 1]:
#                 self.Gi_144 = False
#                 self.Gi_164 = True

#             if self.df['close'][i] < self.df['bb_lower'][i] and self.df['close'][i - 1] <= self.df['bb_lower'][i - 1]:
#                 self.Gi_144 = True
#                 self.Gi_164 = False

#             if self.df['bulls_power'][i] > 0.0 and self.df['bulls_power'][i - 1] > self.df['bulls_power'][i]:
#                 self.Gi_140 = False
#                 self.Gi_160 = True

#             if self.df['bears_power'][i] < 0.0 and self.df['bears_power'][i - 1] < self.df['bears_power'][i]:
#                 self.Gi_140 = True
#                 self.Gi_160 = False

#             if self.df['rsi'][i] > 50.0 and self.df['rsi'][i - 1] < 50.0:
#                 self.Gi_136 = True
#                 self.Gi_156 = False

#             if self.df['rsi'][i] < 50.0 and self.df['rsi'][i - 1] > 50.0:
#                 self.Gi_136 = False
#                 self.Gi_156 = True

#             if self.df['ema_fast'][i] > self.df['ema_slow'][i] and self.df['ema_fast'][i - 1] < self.df['ema_slow'][i - 1]:
#                 self.Gi_132 = True
#                 self.Gi_152 = False

#             if self.df['ema_fast'][i] < self.df['ema_slow'][i] and self.df['ema_fast'][i - 1] > self.df['ema_slow'][i - 1]:
#                 self.Gi_132 = False
#                 self.Gi_152 = True

#             if (self.Gi_132 and self.Gi_136 and self.Gi_144 and self.Gi_140 and self.Gi_148 and self.Gi_172 != 1):
#                 self.df.at[i, 'SuperArrowSignal'] = 1
#                 self.Gi_172 = 1

#             elif (self.Gi_152 and self.Gi_156 and self.Gi_164 and self.Gi_160 and not self.Gi_168 and self.Gi_172 != 2):
#                 self.df.at[i, 'SuperArrowSignal'] = 2
#                 self.Gi_172 = 2
#         return self.df

#     def save_results(self, output_file):
#         self.df['UTC'] = pd.to_datetime(self.df['datetime']) + timedelta(hours=5)
#         self.df['GMT'] = self.df['UTC'] + timedelta(hours=2)
#         self.df.to_csv(output_file, index=False)

#     def run(self, output_file=None) -> pd.DataFrame:
#         self.calculate_indicators()
#         return self.apply_strategy()
#         # self.save_results(output_file)

In [ ]:
class SuperSignalV2Generator:
    def __init__(self, df: pd.DataFrame, file_path=None):
        self.df = df
        self.parameters = {
                'dist': 24,
                'SignalGap': 4,
                'SoundON': True,
                'EmailON': False
            }
        self.initialize_buffers_and_flags()

    def initialize_buffers_and_flags(self):
        self.df['SuperSignalV2'] = 0
        self.flagval1 = 0
        self.flagval2 = 0

    @staticmethod
    def highest(data, length):
        return data.rolling(window=length, min_periods=1).max()

    @staticmethod
    def lowest(data, length):
        return data.rolling(window=length, min_periods=1).min()

    def calculate_signals(self):
        dist = self.parameters['dist']
        for i in range(dist, len(self.df)):
            hhb = self.highest(self.df['high'].iloc[i-dist//2:i+dist//2+1], dist).iloc[-1]
            llb = self.lowest(self.df['low'].iloc[i-dist//2:i+dist//2+1], dist).iloc[-1]

            # Buy signal
            if self.df['high'].iloc[i] == hhb:
                self.df.at[self.df.index[i], 'SuperSignalV2'] = 2

            # Sell signal
            if self.df['low'].iloc[i] == llb:
                self.df.at[self.df.index[i], 'SuperSignalV2'] = 1
        return self.df

    def save_results(self, output_file):
        self.df['UTC'] = pd.to_datetime(self.df['datetime']) + timedelta(hours=5)
        self.df['GMT'] = self.df['UTC'] + timedelta(hours=2)
        self.df.to_csv(output_file, index=False)

    def run(self, output_file=None) -> pd.DataFrame:
        return self.calculate_signals()
        # self.save_results(output_file)

# Example usage


# file_path = 'common/MachineLearningModel/output/five_mins/EURUSD_5_Min_testing_new.csv'
# output_file = 'common/MachineLearningModel/output/supersignalv2.csv'
# dataframe = pd.read_csv(file_path)
# generator = SuperSignalV2Generator(dataframe)
# generator.run(output_file)

In [ ]:
class SuperV3SignalPredictor:
    def __init__(self, data=None, file_path=None, output_file=None):
        self.file_path = file_path
        self.output_file = output_file
        self.data = data
        self.setup_parameters()

    def load_data(self):
        return pd.read_csv(self.file_path)

    def setup_parameters(self):
        # Define parameters
        self.dist1 = 14
        self.dist2 = 21

    def calculate_indicators(self):
        df = self.data
        
        # Calculate highest and lowest values over dist1 and dist2 periods
        df['hhb1'] = df['high'].rolling(window=self.dist1, center=True).max()
        df['llb1'] = df['low'].rolling(window=self.dist1, center=True).min()
        df['hhb'] = df['high'].rolling(window=self.dist2, center=True).max()
        df['llb'] = df['low'].rolling(window=self.dist2, center=True).min()
        
        # Calculate ATR
        df['atr'] = ta.volatility.average_true_range(df['high'], df['low'], df['close'], window=50)

    def generate_signals(self):
        df = self.data

        # Initialize b1, b2, b3, b4
        df['b1'] = np.nan
        df['b2'] = np.nan
        df['b3'] = np.nan
        df['b4'] = np.nan

        # Fill b1, b2, b3, b4 based on conditions
        df.loc[df['high'] == df['hhb'], 'b1'] = df['high'] + df['atr']
        df.loc[df['low'] == df['llb'], 'b2'] = df['low'] - df['atr']
        df.loc[df['high'] == df['hhb1'], 'b3'] = df['high'] + df['atr'] / 2
        df.loc[df['low'] == df['llb1'], 'b4'] = df['low'] - df['atr'] / 2
        df['SuperSignalV3'] = 0
        # Generate signals
        conditions = [
            (df['b1'].notna() & df['b3'].notna(), 2),
            (df['b1'].notna() & df['b3'].isna(), 2),
            (df['b1'].isna() & df['b3'].notna(), -2),
            (df['b2'].notna() & df['b4'].notna(), 1),
            (df['b2'].notna() & df['b4'].isna(), 1),
            (df['b2'].isna() & df['b4'].notna(), -1)
        ]

        # Apply conditions to generate signals
        for condition, signal in conditions:
            df.loc[condition, 'SuperSignalV3'] = signal
        return df

    def adjust_timezones(self):
        df = self.data
        df['UTC'] = pd.to_datetime(df['datetime']) + timedelta(hours=5)
        df['GMT'] = df['UTC'] + timedelta(hours=2)

    def save_output(self):
        self.data.to_csv(self.output_file, index=False)

    def run(self) -> pd.DataFrame:
        self.calculate_indicators()
        data = self.generate_signals()
        # data.to_csv('/tmp/supersignalv3.csv')
        # return data[['SuperSignalV3']]
        return data
        # self.adjust_timezones()
        # self.save_output()



In [ ]:
class TMIndicator:

    def __init__(self,df: pd.DataFrame) -> None:
        self.df = df.copy()
        # Define the parameters
        self.half_length = 10
        self.price_column = 'close'
        self.bands_deviations = 2.4
        self.koeff = 0.0001
        self.interpolate = True
        # Initialize buffers
        self.tm_buffer = np.zeros(len(self.df))
        self.up_buffer = np.zeros(len(self.df))
        self.dn_buffer = np.zeros(len(self.df))
        self.wu_buffer = np.zeros(len(self.df))
        self.wd_buffer = np.zeros(len(self.df))
        self.up_arrow = np.full(len(self.df), np.nan)
        self.dn_arrow = np.full(len(self.df), np.nan)

    # Calculate the TMA and bands
    def calculate_tma(self, half_length, price_column, bands_deviations, koeff):
        full_length = 2.0 * half_length + 1.0
        
        for i in range(len(self.df)):
            sum_val = (half_length + 1) * self.df[price_column].iloc[i]
            sumw = half_length + 1
            for j in range(1, half_length + 1):
                if i + j < len(self.df):
                    sum_val += (half_length - j + 1) * self.df[price_column].iloc[i + j]
                    sumw += (half_length - j + 1)
                if i - j >= 0:
                    sum_val += (half_length - j + 1) * self.df[price_column].iloc[i - j]
                    sumw += (half_length - j + 1)
            
            self.tm_buffer[i] = sum_val / sumw
            
            if i >= half_length:
                diff = self.df[price_column].iloc[i] - self.tm_buffer[i]
                if i == half_length:
                    self.wu_buffer[i] = np.power(diff, 2) if diff >= 0 else 0
                    self.wd_buffer[i] = np.power(diff, 2) if diff < 0 else 0
                else:
                    self.wu_buffer[i] = (self.wu_buffer[i-1] * (full_length - 1) + np.power(diff, 2)) / full_length if diff >= 0 else self.wu_buffer[i-1] * (full_length - 1) / full_length
                    self.wd_buffer[i] = (self.wd_buffer[i-1] * (full_length - 1) + np.power(diff, 2)) / full_length if diff < 0 else self.wd_buffer[i-1] * (full_length - 1) / full_length

                self.up_buffer[i] = self.tm_buffer[i] + bands_deviations * np.sqrt(self.wu_buffer[i])
                self.dn_buffer[i] = self.tm_buffer[i] - bands_deviations * np.sqrt(self.wd_buffer[i])
        
        return self.tm_buffer, self.up_buffer, self.dn_buffer

    def calculate(self):
        return self.calculate_tma(self.half_length, self.price_column, self.bands_deviations, self.koeff)
    def run(self) -> pd.DataFrame:
        # Generate arrows
        self.tm_buffer, self.up_buffer, self.dn_buffer = self.calculate()
        for i in range(1, len(self.df) - 1):
            if self.df['high'].iloc[i+1] > self.up_buffer[i+1] and self.df['close'].iloc[i+1] > self.df['open'].iloc[i+1] and self.df['close'].iloc[i] < self.df['open'].iloc[i]:
                self.up_arrow[i] = self.df['high'].iloc[i] + self.df['close'].rolling(window=20).mean().iloc[i] + self.koeff
            if self.df['low'].iloc[i+1] < self.dn_buffer[i+1] and self.df['close'].iloc[i+1] < self.df['open'].iloc[i+1] and self.df['close'].iloc[i] > self.df['open'].iloc[i]:
                self.dn_arrow[i] = self.df['low'].iloc[i] - self.df['close'].rolling(window=20).mean().iloc[i] - self.koeff
        # return self.df
        decimal_length = len(str(self.df['open'].iloc[0]).split('.')[1])
        
        # Use .loc to avoid the SettingWithCopyWarning
        self.df.loc[:, 'tm_buffer'] = np.round(self.tm_buffer, decimal_length)
        self.df.loc[:, 'up_buffer'] = np.round(self.up_buffer, decimal_length)
        self.df.loc[:, 'dn_buffer'] = np.round(self.dn_buffer, decimal_length)
        self.df.loc[:, 'up_arrow'] = np.round(self.up_arrow, decimal_length)
        self.df.loc[:, 'dn_arrow'] = np.round(self.dn_arrow, decimal_length)
        self.df.loc[:,'TMSignal'] = 0
        # Vectorized computation for SELL_TM and BUY_TM
        self.df.loc[
            (self.df[['open', 'close']].min(axis=1) <= self.df['up_buffer']) & 
            (self.df['up_buffer'] <= self.df[['open', 'close']].max(axis=1)), 
            'TMSignal'
        ] = 2

        # Set TMSignal to 1 if dn_buffer is within the range of open and close
        self.df.loc[
            (self.df[['open', 'close']].min(axis=1) <= self.df['dn_buffer']) & 
            (self.df['dn_buffer'] <= self.df[['open', 'close']].max(axis=1)), 
            'TMSignal'
        ] = 1
        return self.df


In [92]:
# Example usage
file_path = 'common/MachineLearningModel/output/five_mins/EURUSD_5_Min_testing_new.csv'
file_path_1_min = 'common/MachineLearningModel/output/five_mins/EURUSD_1_Min_testing_new.csv'
file_path_15_min = 'common/MachineLearningModel/output/five_mins/EURUSD_15_Min_testing_new.csv'
output_file = 'common/MachineLearningModel/output/newDataFrameOutput.csv'
output_file = 'common/MachineLearningModel/output/newDataFrameOutput_15.csv'
dataframe = pd.read_csv(file_path)
dataframe_1_min = pd.read_csv(file_path_1_min)
dataframe_15_min = pd.read_csv(file_path_15_min)
# dataframe['UTC'] = pd.to_datetime(dataframe['datetime']) + timedelta(hours=5)
# dataframe['GMT'] = dataframe['UTC'] + timedelta(hours=2)
# dataframe_15_min['UTC'] = pd.to_datetime(dataframe_15_min['datetime']) + timedelta(hours=5)
# dataframe_15_min['GMT'] = dataframe_15_min['UTC'] + timedelta(hours=2)
# tested and ok
superv3 = SuperV3SignalPredictor(dataframe.copy()).run()
superv3.to_csv('common/MachineLearningModel/output/superv3.csv')
# tested and ok
superv2 = SuperSignalV2Generator(superv3.copy().reset_index(drop=True)).run()
superv2.to_csv('common/MachineLearningModel/output/superv2.csv')
# tested and ok
binary = BinaryArrowSignalPredictor(superv2.copy().reset_index(drop=True)).run()
binary.to_csv('common/MachineLearningModel/output/binary.csv')
# tested and ok
# superarrow = SuperArrowSignalGenerator(dataframe_15_min.copy().reset_index(drop=True)).run()
# superarrow.to_csv('common/MachineLearningModel/output/superarrow.csv')

# extremebinary = ExtremeBinarySignalPredictor(binary.copy().reset_index(drop=True),dataframe_1_min.copy()).run()
# extremebinary.to_csv('common/MachineLearningModel/output/extremebinary.csv')
# tested and ok
new_dataframe = TMIndicator(binary.copy().reset_index(drop=True)).run()

# new_dataframe.to_csv('common/MachineLearningModel/output/tmindicator.csv')
# # Ensure index is reset after running the functions
# # superv3.reset_index(drop=True, inplace=True)
# # superv2.reset_index(drop=True, inplace=True)
# # binary.reset_index(drop=True, inplace=True)
# # superarrow.reset_index(drop=True, inplace=True)
# # extremebinary.reset_index(drop=True, inplace=True)
# # tmindicator.reset_index(drop=True, inplace=True)

# # new_dataframe = pd.concat([superv3, superv2, binary, superarrow, extremebinary, tmindicator], axis=0)

# new_dataframe.to_csv(output_file, index=False)



In [ ]:
# super arrow
# import pandas as pd
# import numpy as np
# import ta

# # Assuming df is your DataFrame containing 'open', 'high', 'low', 'close', and 'volume' columns.
# # Example: df = pd.read_csv('your_data.csv')
# # Assuming df is your DataFrame containing 'open', 'high', 'low', 'close', and 'volume' columns.
# # Example: df = pd.read_csv('your_data.csv')
# file_path = 'common/MachineLearningModel/output/five_mins/EURUSD_5_Min_testing_new.csv'
# df = pd.read_csv(file_path).reset_index(drop=True)
# # Parameters
# FasterMovingAverage = 5
# SlowerMovingAverage = 12
# RSIPeriod = 12
# MagicFilterPeriod = 1
# BollingerbandsPeriod = 10
# BollingerbandsShift = 0
# BollingerbandsDeviation = 0.5
# BullsPowerPeriod = 50
# BearsPowerPeriod = 50
# Utstup = 10
# Alerts = True

# # Calculate Indicators
# df['ema_fast'] = ta.trend.ema_indicator(df['close'], window=FasterMovingAverage)
# df['ema_slow'] = ta.trend.ema_indicator(df['close'], window=SlowerMovingAverage)
# df['rsi'] = ta.momentum.rsi(df['close'], window=RSIPeriod)
# df['bulls_power'] = df['high'] - ta.trend.ema_indicator(df['close'], window=BullsPowerPeriod)
# df['bears_power'] = df['low'] - ta.trend.ema_indicator(df['close'], window=BearsPowerPeriod)
# bb = ta.volatility.BollingerBands(df['close'], window=BollingerbandsPeriod, window_dev=BollingerbandsDeviation)
# df['bb_upper'] = bb.bollinger_hband()
# df['bb_lower'] = bb.bollinger_lband()

# # Initialize conditions
# Gi_132 = False
# Gi_136 = False
# Gi_140 = False
# Gi_144 = False
# Gi_148 = False
# Gi_152 = False
# Gi_156 = False
# Gi_160 = False
# Gi_164 = False
# Gi_168 = False
# Gi_172 = 0
# Gi_176 = False
# Gi_180 = False

# df['signal_buy'] = np.nan
# df['signal_sell'] = np.nan

# # Loop through data
# for i in range(len(df)-1,1,-1):
#     if i < 10:
#         continue  # Skip initial periods where sufficient data isn't available

#     # Calculate Ld_124
#     Ld_140 = np.sum(np.abs(df['high'][i:i + 10] - df['low'][i:i + 10]))
#     Ld_132 = Ld_140 / 10.0
#     Ld_124 = 100 - 100.0 * ((Ld_132 - 0.0) / 10.0)

#     if Ld_124 >= 0.0:
#         Gi_148 = True
#         Gi_168 = False
#     else:
#         Gi_148 = False
#         Gi_168 = True

#     if df['close'][i] > df['bb_upper'][i] and df['close'][i - 1] >= df['bb_upper'][i - 1]:
#         Gi_144 = False
#         Gi_164 = True

#     if df['close'][i] < df['bb_lower'][i] and df['close'][i - 1] <= df['bb_lower'][i - 1]:
#         Gi_144 = True
#         Gi_164 = False

#     if df['bulls_power'][i] > 0.0 and df['bulls_power'][i - 1] > df['bulls_power'][i]:
#         Gi_140 = False
#         Gi_160 = True

#     if df['bears_power'][i] < 0.0 and df['bears_power'][i - 1] < df['bears_power'][i]:
#         Gi_140 = True
#         Gi_160 = False

#     if df['rsi'][i] > 50.0 and df['rsi'][i - 1] < 50.0:
#         Gi_136 = True
#         Gi_156 = False

#     if df['rsi'][i] < 50.0 and df['rsi'][i - 1] > 50.0:
#         Gi_136 = False
#         Gi_156 = True

#     if df['ema_fast'][i] > df['ema_slow'][i] and df['ema_fast'][i - 1] < df['ema_slow'][i - 1]:
#         Gi_132 = True
#         Gi_152 = False

#     if df['ema_fast'][i] < df['ema_slow'][i] and df['ema_fast'][i - 1] > df['ema_slow'][i - 1]:
#         Gi_132 = False
#         Gi_152 = True

#     # Check buy conditions
#     if (Gi_132 and Gi_136 and Gi_144 and Gi_140 and Gi_148 and Gi_172 != 1):
#         df.at[i, 'signal_buy'] = df['low'][i] - (Utstup * df['low'][i])
#         Gi_172 = 1

#     # Check sell conditions
#     elif (Gi_152 and Gi_156 and Gi_164 and Gi_160 and not Gi_168 and Gi_172 != 2):
#         df.at[i, 'signal_sell'] = df['high'][i] + (Utstup * df['high'][i])
#         Gi_172 = 2

# df['UTC'] = pd.to_datetime(df['datetime']) + timedelta(hours=5)
# df['GMT'] = df['UTC'] + timedelta(hours=2)
# output_file = 'common/MachineLearningModel/output/newsuperarrow.csv'
# df.to_csv(output_file, index=False)


In [ ]:
# super signal v2
# import pandas as pd
# import numpy as np
# from datetime import timedelta

# class SuperSignalV2Generator:
#     def __init__(self, df: pd.DataFrame, file_path=None):
#         self.df = df
#         self.parameters = {
#                 'dist': 24,
#                 'SignalGap': 4,
#                 'SoundON': True,
#                 'EmailON': False
#             }
#         self.initialize_buffers_and_flags()

#     def initialize_buffers_and_flags(self):
#         self.df['SuperSignalV2'] = 0
#         self.flagval1 = 0
#         self.flagval2 = 0

#     @staticmethod
#     def highest(data, length):
#         return data.rolling(window=length, min_periods=1).max()

#     @staticmethod
#     def lowest(data, length):
#         return data.rolling(window=length, min_periods=1).min()

#     def calculate_signals(self):
#         dist = self.parameters['dist']
#         for i in range(dist, len(self.df)):
#             hhb = self.highest(self.df['high'].iloc[i-dist//2:i+dist//2+1], dist).iloc[-1]
#             llb = self.lowest(self.df['low'].iloc[i-dist//2:i+dist//2+1], dist).iloc[-1]

#             # Buy signal
#             if self.df['high'].iloc[i] == hhb:
#                 self.df.at[self.df.index[i], 'SuperSignalV2'] = 2

#             # Sell signal
#             if self.df['low'].iloc[i] == llb:
#                 self.df.at[self.df.index[i], 'SuperSignalV2'] = 1
#         return self.df

#     def save_results(self, output_file):
#         self.df['UTC'] = pd.to_datetime(self.df['datetime']) + timedelta(hours=5)
#         self.df['GMT'] = self.df['UTC'] + timedelta(hours=2)
#         self.df.to_csv(output_file, index=False)

#     def run(self, output_file):
#         return self.calculate_signals()
#         # self.save_results(output_file)

# # Example usage


# file_path = 'common/MachineLearningModel/output/five_mins/EURUSD_5_Min_testing_new.csv'
# output_file = 'common/MachineLearningModel/output/supersignalv2.csv'
# dataframe = pd.read_csv(file_path)
# generator = SuperSignalV2Generator(dataframe)
# generator.run(output_file)


In [ ]:
def calculate(dataframe: pd.DataFrame) -> pd.DataFrame:
    superv3 = SuperV3SignalPredictor(dataframe.copy()).run()
    # superv3.to_csv('common/MachineLearningModel/output/superv3.csv')
    # tested and ok
    superv2 = SuperSignalV2Generator(superv3.reset_index(drop=True)).run()
    # superv2.to_csv('common/MachineLearningModel/output/superv2.csv')
    # tested and ok
    binary = BinaryArrowSignalPredictor(superv2.reset_index(drop=True)).run()
    # binary.to_csv('common/MachineLearningModel/output/binary.csv')
    # tested and ok
    # superarrow = SuperArrowSignalGenerator(dataframe_15_min.copy().reset_index(drop=True)).run()
    # superarrow.to_csv('common/MachineLearningModel/output/superarrow.csv')

    # extremebinary = ExtremeBinarySignalPredictor(binary.copy().reset_index(drop=True),dataframe_1_min.copy()).run()
    # extremebinary.to_csv('common/MachineLearningModel/output/extremebinary.csv')
    # tested and ok
    new_dataframe = TMIndicator(binary.reset_index(drop=True)).run()
    custom_Columns = ['datetime','symbol','open','high','low','close','next_close','volume','SuperSignalV3','SuperSignalV2','BinaryArrow','TMSignal']
    new_dataframe['next_close'] = new_dataframe['close'].shift(-3)
    new_dataframe = new_dataframe[custom_Columns]
    predit_signals = ['SuperSignalV3','SuperSignalV2','BinaryArrow','TMSignal']
    new_dataframe[predit_signals] = new_dataframe[predit_signals].shift(1)
    all_signals_zero = (new_dataframe[predit_signals] == 0).all(axis=1)
    
    new_dataframe['Prediction'] = np.where(
                        all_signals_zero, 0,
                        np.where(new_dataframe['open'] < new_dataframe['next_close'], 1,
                                np.where(new_dataframe['open'] > new_dataframe['next_close'], 2, 0))
                    ).astype('int32')
    # new_dataframe[predit_signals].dropna(inplace=True)
    # new_dataframe.reset_index(drop=True,inplace=True)
    
    return new_dataframe

In [ ]:
def process_files(file_paths):
    pd_data = []
    for file_path in file_paths:
        df = pd.read_csv(file_path)
        cal = calculate(df)
        pd_data.append(cal)
    return pd.concat(pd_data)

first_list = ['EURUSD', 'EURCAD', 'EURJPY', 'EURGBP', 'EURAUD'] # 
sc_list = ['EURUSD', 'EURCAD', 'EURJPY', 'EURGBP', 'USDCAD', 'USDJPY']
th_list = ['EURAUD', 'EURUSD', 'EURCAD', 'EURJPY', 'EURGBP', 'USDCAD', 'USDJPY']

file_paths = []

for curr in first_list:
    file_paths.append(f"common/MachineLearningModel/output/five_mins/{curr}_5_Min.csv")

for curr in sc_list:
    file_paths.append(f'common/MachineLearningModel/output/five_mins/{curr}_5_Min_1.csv')

for curr in th_list:
    file_paths.append(f'common/MachineLearningModel/output/five_mins/{curr}_5_Min_2.csv')

for curr in th_list:
    file_paths.append(f'common/MachineLearningModel/output/five_mins/{curr}_5_Min_3.csv')
for curr in th_list:
    file_paths.append(f'common/MachineLearningModel/output/five_mins/{curr}_5_Min_4.csv')

data = process_files(file_paths)

In [85]:
predit_signals = ['SuperSignalV3','SuperSignalV2','BinaryArrow','TMSignal']
at_least_two_1_or_2 = (data[predit_signals].isin([1, 2])).sum(axis=1) >= 2
all_signals_zero = (data[predit_signals] == 0).all(axis=1)
# Combine the conditions (if you want both conditions to be true)
combined_condition = all_signals_zero
data['Prediction'] = np.where(
    all_signals_zero, 0,
    np.where(
        at_least_two_1_or_2 & (data['open'] < data['next_close']), 1,
        np.where(
            at_least_two_1_or_2 & (data['open'] > data['next_close']), 2,
            0
        )
    )
).astype('int32')

In [87]:

print(data.iloc[:,8:].tail())
print(data.shape)

      SuperSignalV3  SuperSignalV2  BinaryArrow  TMSignal  Prediction
5337            0.0            0.0          0.0       0.0           0
5338            0.0            0.0          0.0       0.0           0
5339            0.0            0.0          0.0       0.0           0
5340            0.0            0.0          0.0       0.0           0
5341            0.0            0.0          0.0       0.0           0
(212926, 13)


In [88]:
le = LabelEncoder()
le.fit_transform(data['Prediction'])
print(le.classes_)

[0 1 2]


In [89]:
print(data['Prediction'].value_counts())
X = data.iloc[:,8:-3]
y = data.iloc[:, -1]
print(data.columns)
print(X.columns)
print(X.count())
# print(y.head())
X_train, X_test, y_train, y_test =train_test_split(
  X, y, test_size = 0.35,train_size=0.65, shuffle=False)

Prediction
0    197903
2      7522
1      7501
Name: count, dtype: int64
Index(['datetime', 'symbol', 'open', 'high', 'low', 'close', 'next_close',
       'volume', 'SuperSignalV3', 'SuperSignalV2', 'BinaryArrow', 'TMSignal',
       'Prediction'],
      dtype='object')
Index(['SuperSignalV3', 'SuperSignalV2'], dtype='object')
SuperSignalV3    212894
SuperSignalV2    212894
dtype: int64


In [ ]:
# Initialize XGBoost classifier
xgb_model = XGBClassifier(booster="gbtree",max_depth=14,min_child_weight = 2)
# Train the model
xgb_model.fit(X_train, y_train)

# Make predictions on the test set
preds = xgb_model.predict(X_test)

# Evaluate the model
print(f"Accuracy on train data by XGBoost Classifier\
: {accuracy_score(y_train, xgb_model.predict(X_train))*100}")
 
print(f"Accuracy on test data by XGBoost Classifier\
: {accuracy_score(y_test, preds)*100}")

In [ ]:

rf_model = RandomForestClassifier()
# Train the model
rf_model.fit(X_train, y_train)

# Make predictions on the test set  
preds = rf_model.predict(X_test)

# Evaluate the model
print(f"Accuracy on train data by RandomForest Classifier\
: {accuracy_score(y_train, rf_model.predict(X_train))*100}")
 
print(f"Accuracy on test data by RandomForest Classifier\
: {accuracy_score(y_test, preds)*100}")

In [ ]:
final_rf_model = RandomForestClassifier()
final_rf_model.fit(X, y)

In [ ]:
final_xgb_model = XGBClassifier(booster="gbtree",max_depth=14,min_child_weight = 2)
final_xgb_model.fit(X, y)

In [90]:
df = pd.read_csv('/home/magesh/TrandingProjects/Project/backend/dolphin/common/MachineLearningModel/output/five_mins/EURUSD_5_Min_testing_new.csv')
cal = calculate(df)
test_data = cal.iloc[:,8:-3]
cal['rf_predict'] = rf_model.predict(test_data)
cal['xgb_predict'] = xgb_model.predict(test_data)
cal.to_csv('/tmp/newoneresultsusd.csv')